# Build Reusable Data Products




In [ ]:
import os
import sys
from pathlib import Path
from pyspark.sql import functions as F

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark
from src.lake import GOLD, read_delta, write_delta

spark = create_spark("data-products")

integrated = read_delta(spark, GOLD / "integrated_taxi_trips")
integrated.createOrReplaceTempView("integrated_taxi_trips")

def with_metadata(df, schema_ver="1.0"):
    now = F.current_timestamp()
    return (
        df.withColumn("_data_source", F.lit("gold/integrated_taxi_trips"))
          .withColumn("_created_at", now)
          .withColumn("_refreshed_at", now)
          .withColumn("_schema_version", F.lit(schema_ver))
    )

:: loading settings :: url = jar:file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/samuelflodin/.ivy2/cache
The jars for the packages stored in: /Users/samuelflodin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3be2ae19-4c7e-4c64-86c8-89504667f401;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 90ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |

## Product 1: Daily Mobility Summary

In [9]:
query_prod_1 = """
SELECT
    pickup_date,
    pickup_borough,
    COUNT(*) AS total_trips,
    SUM(passenger_count) AS total_passengers,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM integrated_taxi_trips
WHERE pickup_borough != 'UNKNOWN' AND pickup_date IS NOT NULL
GROUP BY pickup_date, pickup_borough
ORDER BY pickup_date, total_trips DESC
"""
df_prod_1 = with_metadata(spark.sql(query_prod_1), "1.0")
write_delta(df_prod_1, GOLD / "data_products" / "daily_borough_mobility", partition_by=["pickup_date"])
print("Saved Product 1: daily_borough_mobility")

Saved Product 1: daily_borough_mobility


## Product 2: Taxi Zone Monthly Demand

In [10]:
query_prod_2 = """
WITH monthly_zone_trips AS (
    SELECT
        TRUNC(pickup_date, 'MM') AS trip_month,
        pickup_location_id,
        pickup_borough,
        pickup_zone,
        pickup_date
    FROM integrated_taxi_trips
    WHERE pickup_zone != 'UNKNOWN' AND pickup_date IS NOT NULL
)
SELECT
    trip_month,
    pickup_location_id,
    pickup_borough,
    pickup_zone,
    COUNT(*) AS total_trips,
    COUNT(DISTINCT pickup_date) AS active_days,
    ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date), 2) AS avg_daily_trips
FROM monthly_zone_trips
GROUP BY trip_month, pickup_location_id, pickup_borough, pickup_zone
"""
df_prod_2 = with_metadata(spark.sql(query_prod_2), "1.0")
write_delta(df_prod_2, GOLD / "data_products" / "taxi_zone_monthly_demand", partition_by=["trip_month"])
print("Saved Product 2: taxi_zone_monthly_demand")

Saved Product 2: taxi_zone_monthly_demand


## Product 3: Weather Impact Summary

In [ ]:
query_prod_3 = """
WITH binned_weather AS (
    SELECT
        trip_distance,
        CASE
            WHEN temperature_c IS NULL THEN 'Unknown'
            WHEN temperature_c < 0 THEN 'Freezing (<0°C)'
            WHEN temperature_c BETWEEN 0 AND 10 THEN 'Cold (0°C to 10°C)'
            WHEN temperature_c BETWEEN 10.01 AND 20 THEN 'Moderate (10°C to 20°C)'
            ELSE 'Warm (>20°C)'
        END AS temp_category,
        CASE
            WHEN wind_speed_ms IS NULL THEN 'Unknown'
            WHEN wind_speed_ms < 2 THEN 'Calm (<2 m/s)'
            WHEN wind_speed_ms BETWEEN 2 AND 6 THEN 'Moderate Wind (2-6 m/s)'
            ELSE 'High Wind (>6 m/s)'
        END AS wind_category
    FROM integrated_taxi_trips
    WHERE trip_distance > 0 AND trip_distance < 100
)
SELECT
    temp_category,
    wind_category,
    COUNT(*) AS trip_count,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles
FROM binned_weather
GROUP BY temp_category, wind_category
ORDER BY temp_category, wind_category
"""
df_prod_3 = with_metadata(spark.sql(query_prod_3), "1.0")
write_delta(df_prod_3, GOLD / "data_products" / "weather_impact_summary")
print("Saved Product 3: weather_impact_summary")

Saved Product 3: weather_impact_summary


## Product 4: Air Quality Demand Summary

In [ ]:
query_prod_4 = """
WITH rounded_pm25 AS (
    SELECT
        ROUND(pm25, 0) AS pm25_level,
        pickup_date,
        pickup_hour
    FROM integrated_taxi_trips
    WHERE pm25 IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
)
SELECT
    pm25_level,
    COUNT(*) AS trips,
    COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS observed_hours,
    ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT struct(pickup_date, pickup_hour)), 2) AS trips_per_hour
FROM rounded_pm25
GROUP BY pm25_level
ORDER BY pm25_level DESC
"""
df_prod_4 = with_metadata(spark.sql(query_prod_4), "1.0")
write_delta(df_prod_4, GOLD / "data_products" / "air_quality_demand_summary")
print("Saved Product 4: air_quality_demand_summary")

26/09/20 14:13:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Saved Product 4: air_quality_demand_summary


Product 5: Taxi Zone Weather Sensitivity Summary

In [13]:
query_prod_5 = """
WITH trips_with_weather AS (
    SELECT 
        pickup_zone,
        pickup_date,
        pickup_hour,
        NTILE(4) OVER (
            PARTITION BY pickup_zone
            ORDER BY (10 * sqrt(wind_speed_ms) - wind_speed_ms + 10.5) * (33 - temperature_c)
        ) AS weather_condition
    FROM integrated_taxi_trips
    WHERE pickup_zone IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
),
hourly_demand AS (
    SELECT 
        pickup_zone,
        weather_condition,
        COUNT(1) / COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS trips_per_hour
    FROM trips_with_weather
    GROUP BY pickup_zone, weather_condition
),
pivoted AS (
    SELECT * FROM hourly_demand
    PIVOT (
        ROUND(AVG(trips_per_hour), 2)
        FOR weather_condition IN (1 AS coldest, 2 AS cool, 3 AS warm, 4 AS warmest)
    )
)
SELECT 
    pickup_zone,
    coldest, cool, warm, warmest,
    ROUND(((GREATEST(coldest, cool, warm, warmest) - LEAST(coldest, cool, warm, warmest)) / ((coldest + cool + warm + warmest) / 4.0)) * 100, 2) AS pct_variation
FROM pivoted
WHERE (coldest + cool + warm + warmest) / 4.0 >= 10
ORDER BY pct_variation DESC
"""
df_prod_5 = with_metadata(spark.sql(query_prod_5), "1.0")

write_delta(df_prod_5, GOLD / "data_products" / "zone_weather_sensitivity")
print("Saved Product 5: zone_weather_sensitivity")

Saved Product 5: zone_weather_sensitivity
